# Reported Flood-Affected Locations Along the Euphrates in Syria

Overview

This notebook maps the flood-affected locations currently recorded in the project CSV dataset for the Euphrates corridor in Syria.

The current dataset contains six reported locations associated with one flood event dated 25 May 2026. The locations are recorded in Aleppo and Deir-ez-Zor Governorates. Ar-Raqqa is included as geographical context for the Euphrates corridor but does not contain a reported location in the current dataset.

The CSV dataset is maintained as an updateable project dataset. Therefore, this map represents the records available at the time the notebook is executed and may change when new locations or source information are added.

このNotebookでは、プロジェクトのCSVデータに現在収録されている、シリアのユーフラテス川流域における洪水被害報告地点を地図上に表示します。

現在のデータには、2026年5月25日の1件の洪水事象に関連する6地点が収録されています。報告地点はAleppo県およびDeir-ez-Zor県に位置します。Ar-Raqqa県はユーフラテス川流域の地理的背景として表示しますが、現在のCSVには同県内の報告地点は収録されていません。

このCSVは、今後の情報追加を前提とした更新可能なプロジェクトデータです。そのため、この地図はNotebook実行時点で収録されている地点を示しており、新しい地点や出典情報の追加に伴って更新されます。

Objectives

- Read and validate the current flood-affected location records
- Convert latitude and longitude records into spatial point data
- Verify reported governorates through a spatial join
- Extract and display the Euphrates River from the source river dataset
- Display the Euphrates corridor governorates as geographical context
- Export validated point data as a derived GeoPackage
- Create an interactive map with information, source and legend panels

- 現在の洪水被害報告地点データを読み込み、検証する
- 緯度・経度から地点の空間データを作成する
- 空間結合により、CSVに記録された県名を検証する
- 河川データからユーフラテス川を抽出して表示する
- 地理的背景としてユーフラテス川流域の県境を表示する
- 検証済み地点データを派生GeoPackageとして保存する
- 説明、出典および凡例を備えたインタラクティブ地図を作成する

Workflow

#### English

1. Read the flood-affected location CSV
2. Validate identifiers, dates, coordinates and required fields
3. Create a GeoDataFrame from the reported coordinates
4. Read and validate the administrative boundary, river and water-body datasets
5. Reproject the river data to WGS 84
6. Extract the Euphrates River features
7. Assign governorate attributes to the reported locations through a spatial join
8. Compare the reported and spatially assigned governorates
9. Save the validated locations as a derived GeoPackage
10. Create and export the interactive map

#### 日本語

1. 洪水被害報告地点CSVを読み込む
2. 識別子、日付、座標および必須項目を検証する
3. 報告座標からGeoDataFrameを作成する
4. 行政界、河川および水域データを読み込み、検証する
5. 河川データをWGS 84へ変換する
6. ユーフラテス川の地物を抽出する
7. 空間結合により報告地点へ県情報を付与する
8. CSV記載県と空間判定県を比較する
9. 検証済み地点を派生GeoPackageとして保存する
10. インタラクティブ地図を作成し、保存する

Data

Flood-affected location data:
- climap_euphrates_flood_affected_locations.csv
- Source: User-maintained project dataset

Administrative boundary data:
- syr_admin1.geojson
- Source: HDX OCHA, Syria subnational administrative boundaries

Hydrography data:
- syr_rivers_3857.geojson
- syr_lakes_4326.geojson
- Source: OpenStreetMap contributors

Data Scope and Limitations

- The current CSV contains six reported locations linked to one event
- The records currently cover Aleppo and Deir-ez-Zor Governorates
- Ar-Raqqa is displayed only as geographical context
- The dataset represents reported locations and does not describe the complete spatial extent of flooding
- Source organisations are recorded, but detailed source titles, URLs and access dates remain to be added
- The map should not be interpreted as a comprehensive inventory of all flood-affected locations in Syria

- 現在のCSVには、1件の事象に関連する6地点が収録されています
- 現在の報告地点はAleppo県およびDeir-ez-Zor県に位置します
- Ar-Raqqa県は地理的背景としてのみ表示します
- このデータは報告地点を示すものであり、洪水の面的な広がりを示すものではありません
- 情報提供組織は記録されていますが、資料名、URLおよび閲覧日は今後追加する必要があります
- この地図は、シリア国内のすべての洪水被害地点を網羅するものではありません

Technologies

- Python
- Pandas
- GeoPandas
- Folium
- Shapely

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

# Standard library
# 標準ライブラリ

from pathlib import Path


# Tabular and spatial data processing
# 表形式データおよび空間データの処理

import geopandas as gpd
import pandas as pd


# Interactive web mapping
# インタラクティブWeb地図の作成

import folium

from branca.element import Element
from folium.plugins import Fullscreen, MiniMap

In [ ]:
# 2
# Define the project input and output paths
# プロジェクトの入力・出力パスを定義する

PROJECT_ROOT = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts"
)

DATA_DIR = (
    PROJECT_ROOT
    / "02_DATA"
)

PROJECT_DIR = (
    PROJECT_ROOT
    / "01_PROJECTS"
    / "05_CLIMATE_HAZARD_MAPPING"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "outputs"
)

flood_locations_path = (
    DATA_DIR
    / "TABULAR"
    / "climap_euphrates_flood_affected_locations.csv"
)

admin1_path = (
    DATA_DIR
    / "VECTOR"
    / "syr_admin1.geojson"
)

rivers_path = (
    DATA_DIR
    / "VECTOR"
    / "syr_rivers_3857.geojson"
)

water_bodies_path = (
    DATA_DIR
    / "VECTOR"
    / "syr_lakes_4326.geojson"
)

validated_locations_path = (
    OUTPUT_DIR
    / "euphrates_flood_affected_locations_validated.gpkg"
)

output_path = (
    PROJECT_DIR
    / "01_syria_euphrates_flood_affected_locations.html"
)

required_input_paths = {
    "Flood-affected locations": flood_locations_path,
    "Governorate boundaries": admin1_path,
    "River features": rivers_path,
    "Mapped water bodies": water_bodies_path,
}

for dataset_name, dataset_path in required_input_paths.items():

    if not dataset_path.exists():

        raise FileNotFoundError(
            f"{dataset_name} was not found: "
            f"{dataset_path}"
        )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Input path validation: passed")

for dataset_name, dataset_path in required_input_paths.items():

    print(
        f"{dataset_name}: {dataset_path}"
    )

print(
    f"Derived output directory: {OUTPUT_DIR}"
)

In [ ]:
# 3
# Read the current flood-affected location records
# 現在の洪水被害報告地点データを読み込む

flood_records = pd.read_csv(
    flood_locations_path,
    dtype={
        "site_id": "string",
        "event_id": "string",
        "governorate_reported": "string",
        "location": "string",
        "source_organisation": "string",
        "source_title": "string",
        "source_url": "string",
        "accessed_date": "string",
        "notes": "string",
    },
)

if flood_records.empty:

    raise ValueError(
        "The flood-affected location CSV contains no records."
    )

print(
    f"Flood-affected location records: "
    f"{len(flood_records):,}"
)

print(
    f"CSV columns: {len(flood_records.columns):,}"
)

print(
    flood_records.head()
)

In [ ]:
# 4
# Validate the CSV schema and required values
# CSVの列構成と必須項目を検証する

expected_columns = [
    "site_id",
    "event_id",
    "event_date",
    "governorate_reported",
    "location",
    "latitude",
    "longitude",
    "source_organisation",
    "source_title",
    "source_url",
    "accessed_date",
    "notes",
]

required_columns = [
    "site_id",
    "event_id",
    "event_date",
    "governorate_reported",
    "location",
    "latitude",
    "longitude",
    "source_organisation",
]

missing_columns = (
    set(expected_columns)
    - set(flood_records.columns)
)

unexpected_columns = (
    set(flood_records.columns)
    - set(expected_columns)
)

if missing_columns:

    raise ValueError(
        "The flood-affected location CSV is missing columns: "
        f"{sorted(missing_columns)}"
    )

if unexpected_columns:

    raise ValueError(
        "The flood-affected location CSV contains unexpected columns: "
        f"{sorted(unexpected_columns)}"
    )

missing_required_values = {}

for column_name in required_columns:

    column_values = flood_records[
        column_name
    ]

    missing_mask = (
        column_values.isna()
        | column_values.astype("string").str.strip().eq("")
    )

    missing_count = int(
        missing_mask.sum()
    )

    if missing_count > 0:

        missing_required_values[
            column_name
        ] = missing_count

if missing_required_values:

    raise ValueError(
        "Required CSV values are missing: "
        f"{missing_required_values}"
    )

print("CSV schema validation: passed")

print(
    f"Required columns validated: "
    f"{len(required_columns):,}"
)

print(
    f"Optional source-detail fields currently incomplete: "
    f"{flood_records['source_url'].isna().sum():,}"
)

In [ ]:
# 5
# Validate identifiers, dates and coordinates
# 識別子、日付および座標を検証する

text_columns = [
    "site_id",
    "event_id",
    "governorate_reported",
    "location",
    "source_organisation",
]

for column_name in text_columns:

    flood_records[
        column_name
    ] = (
        flood_records[
            column_name
        ]
        .str.strip()
    )

flood_records[
    "event_date"
] = pd.to_datetime(
    flood_records[
        "event_date"
    ],
    format="%Y-%m-%d",
    errors="coerce",
)

if flood_records[
    "event_date"
].isna().any():

    raise ValueError(
        "One or more event dates are invalid."
    )

flood_records[
    "latitude"
] = pd.to_numeric(
    flood_records[
        "latitude"
    ],
    errors="coerce",
)

flood_records[
    "longitude"
] = pd.to_numeric(
    flood_records[
        "longitude"
    ],
    errors="coerce",
)

if flood_records[
    [
        "latitude",
        "longitude",
    ]
].isna().any().any():

    raise ValueError(
        "One or more coordinates are not numeric."
    )

invalid_latitude = (
    ~flood_records[
        "latitude"
    ].between(
        -90,
        90,
    )
)

invalid_longitude = (
    ~flood_records[
        "longitude"
    ].between(
        -180,
        180,
    )
)

if invalid_latitude.any():

    raise ValueError(
        "One or more latitude values are outside "
        "the valid range."
    )

if invalid_longitude.any():

    raise ValueError(
        "One or more longitude values are outside "
        "the valid range."
    )

valid_site_id = (
    flood_records[
        "site_id"
    ].str.fullmatch(
        r"FLD-\d{4}-\d{3}-\d{2}"
    )
)

valid_event_id = (
    flood_records[
        "event_id"
    ].str.fullmatch(
        r"FLD-\d{4}-\d{3}"
    )
)

if not valid_site_id.all():

    raise ValueError(
        "One or more site_id values do not follow "
        "the required format."
    )

if not valid_event_id.all():

    raise ValueError(
        "One or more event_id values do not follow "
        "the required format."
    )

if not flood_records[
    "site_id"
].is_unique:

    duplicate_site_ids = (
        flood_records.loc[
            flood_records[
                "site_id"
            ].duplicated(
                keep=False
            ),
            "site_id",
        ]
        .tolist()
    )

    raise ValueError(
        "Duplicate site_id values were found: "
        f"{duplicate_site_ids}"
    )

print("Identifier, date and coordinate validation: passed")

print(
    "Event-date range:",
    flood_records[
        "event_date"
    ].min().date(),
    "to",
    flood_records[
        "event_date"
    ].max().date(),
)

print(
    "Latitude range:",
    f"{flood_records['latitude'].min():.7f}",
    "to",
    f"{flood_records['latitude'].max():.7f}",
)

print(
    "Longitude range:",
    f"{flood_records['longitude'].min():.7f}",
    "to",
    f"{flood_records['longitude'].max():.7f}",
)

In [ ]:
# 6
# Validate duplicate locations and summarise the records
# 重複地点を検証し、現在の収録内容を集計する

duplicate_location_mask = (
    flood_records.duplicated(
        subset=[
            "event_id",
            "location",
            "latitude",
            "longitude",
        ],
        keep=False,
    )
)

if duplicate_location_mask.any():

    duplicate_locations = (
        flood_records.loc[
            duplicate_location_mask,
            [
                "site_id",
                "event_id",
                "location",
                "latitude",
                "longitude",
            ],
        ]
    )

    raise ValueError(
        "Duplicate event-location records were found:\n"
        f"{duplicate_locations}"
    )

event_count = (
    flood_records[
        "event_id"
    ].nunique()
)

location_count = (
    flood_records[
        "site_id"
    ].nunique()
)

governorate_summary = (
    flood_records
    .groupby(
        "governorate_reported"
    )
    .agg(
        reported_locations=(
            "site_id",
            "nunique",
        ),
        event_count=(
            "event_id",
            "nunique",
        ),
    )
    .sort_index()
)

source_summary = (
    flood_records[
        "source_organisation"
    ]
    .value_counts()
    .sort_index()
)

print(
    f"Recorded flood events: {event_count:,}"
)

print(
    f"Reported locations: {location_count:,}"
)

print(
    "\nReported locations by governorate:"
)

print(
    governorate_summary
)

print(
    "\nRecords by source organisation:"
)

print(
    source_summary
)

In [ ]:
# 7
# Convert the reported coordinates into spatial point data
# 報告座標を空間ポイントデータへ変換する

flood_locations = gpd.GeoDataFrame(
    flood_records.copy(),
    geometry=gpd.points_from_xy(
        flood_records[
            "longitude"
        ],
        flood_records[
            "latitude"
        ],
    ),
    crs="EPSG:4326",
)

if flood_locations.crs is None:

    raise ValueError(
        "The flood-location GeoDataFrame has no defined CRS."
    )

if flood_locations.crs.to_epsg() != 4326:

    raise ValueError(
        "The flood-location GeoDataFrame was expected "
        "to use EPSG:4326."
    )

if flood_locations.geometry.is_empty.any():

    raise ValueError(
        "One or more flood-location geometries are empty."
    )

if not flood_locations.geometry.is_valid.all():

    raise ValueError(
        "One or more flood-location geometries are invalid."
    )

if not flood_locations.geom_type.eq(
    "Point"
).all():

    raise ValueError(
        "The flood-location geometries must all be points."
    )

print(
    f"Flood-location features: "
    f"{len(flood_locations):,}"
)

print(
    f"Flood-location CRS: "
    f"{flood_locations.crs}"
)

print(
    flood_locations[
        [
            "site_id",
            "event_id",
            "event_date",
            "governorate_reported",
            "location",
            "source_organisation",
            "geometry",
        ]
    ]
)

In [ ]:
# 8
# Read the administrative boundary, river and water-body datasets
# 行政界、河川および水域データを読み込む

admin1 = gpd.read_file(
    admin1_path
)

rivers = gpd.read_file(
    rivers_path
)

water_bodies = gpd.read_file(
    water_bodies_path
)

print(
    f"Governorate features: {len(admin1):,}"
)

print(
    f"River features: {len(rivers):,}"
)

print(
    f"Water-body features: {len(water_bodies):,}"
)

print(
    f"Governorate CRS: {admin1.crs}"
)

print(
    f"River CRS: {rivers.crs}"
)

print(
    f"Water-body CRS: {water_bodies.crs}"
)

In [ ]:
# 9
# Validate the spatial datasets and required attributes
# 空間データと必須属性を検証する

spatial_datasets = {
    "Governorate boundaries": admin1,
    "River features": rivers,
    "Water-body features": water_bodies,
}

required_spatial_columns = {
    "Governorate boundaries": {
        "adm1_name",
        "adm1_pcode",
        "center_lat",
        "center_lon",
        "geometry",
    },
    "River features": {
        "name:en",
        "waterway",
        "geometry",
    },
    "Water-body features": {
        "water",
        "natural",
        "geometry",
    },
}

for dataset_name, dataset in spatial_datasets.items():

    if dataset.empty:

        raise ValueError(
            f"{dataset_name} contains no features."
        )

    if dataset.crs is None:

        raise ValueError(
            f"{dataset_name} has no defined CRS."
        )

    missing_dataset_columns = (
        required_spatial_columns[
            dataset_name
        ]
        - set(dataset.columns)
    )

    if missing_dataset_columns:

        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{sorted(missing_dataset_columns)}"
        )

    if dataset.geometry.isna().any():

        raise ValueError(
            f"{dataset_name} contains missing geometries."
        )

    if dataset.geometry.is_empty.any():

        raise ValueError(
            f"{dataset_name} contains empty geometries."
        )

    if not dataset.geometry.is_valid.all():

        invalid_geometry_count = int(
            (~dataset.geometry.is_valid).sum()
        )

        raise ValueError(
            f"{dataset_name} contains "
            f"{invalid_geometry_count:,} invalid geometries."
        )

    print(
        f"{dataset_name} validation: passed"
    )

    print(
        dataset.geom_type.value_counts()
    )

In [ ]:
# 10
# Reproject the spatial datasets to WGS 84
# 空間データをWGS 84へ変換する

WEB_CRS = "EPSG:4326"

admin1_web = admin1.to_crs(
    WEB_CRS
)

rivers_web = rivers.to_crs(
    WEB_CRS
)

water_bodies_web = water_bodies.to_crs(
    WEB_CRS
)

web_datasets = {
    "Governorate boundaries": admin1_web,
    "River features": rivers_web,
    "Water-body features": water_bodies_web,
}

for dataset_name, dataset in web_datasets.items():

    if dataset.crs.to_epsg() != 4326:

        raise ValueError(
            f"{dataset_name} was not correctly "
            "reprojected to EPSG:4326."
        )

    print(
        f"{dataset_name} CRS: {dataset.crs}"
    )

In [ ]:
# 11
# Extract the Euphrates River features
# 河川データからユーフラテス川を抽出する

river_name_english = (
    rivers_web[
        "name:en"
    ]
    .fillna("")
    .astype("string")
    .str.strip()
    .str.casefold()
)

euphrates_river = (
    rivers_web.loc[
        river_name_english.eq(
            "euphrates"
        )
    ]
    .copy()
)

if euphrates_river.empty:

    raise ValueError(
        "No Euphrates River features were found "
        "in the source river dataset."
    )

if euphrates_river.crs.to_epsg() != 4326:

    raise ValueError(
        "The extracted Euphrates River features "
        "do not use EPSG:4326."
    )

if not euphrates_river.geometry.is_valid.all():

    raise ValueError(
        "The extracted Euphrates River features "
        "contain invalid geometries."
    )

print(
    f"Extracted Euphrates River features: "
    f"{len(euphrates_river):,}"
)

print(
    euphrates_river[
        [
            "name",
            "name:en",
            "waterway",
            "source",
        ]
    ]
)

In [ ]:
# 12
# Select the Euphrates corridor governorates
# ユーフラテス川流域の対象県を抽出する

EUPHRATES_CORRIDOR_GOVERNORATES = [
    "Aleppo",
    "Ar-Raqqa",
    "Deir-ez-Zor",
]

available_governorates = set(
    admin1_web[
        "adm1_name"
    ]
)

missing_corridor_governorates = (
    set(
        EUPHRATES_CORRIDOR_GOVERNORATES
    )
    - available_governorates
)

if missing_corridor_governorates:

    raise ValueError(
        "The corridor governorates were not found: "
        f"{sorted(missing_corridor_governorates)}"
    )

euphrates_corridor_admin = (
    admin1_web.loc[
        admin1_web[
            "adm1_name"
        ].isin(
            EUPHRATES_CORRIDOR_GOVERNORATES
        ),
        [
            "adm1_name",
            "adm1_pcode",
            "center_lat",
            "center_lon",
            "geometry",
        ],
    ]
    .copy()
)

if len(euphrates_corridor_admin) != len(
    EUPHRATES_CORRIDOR_GOVERNORATES
):

    raise ValueError(
        "The number of selected corridor governorates "
        "does not match the expected count."
    )

print(
    f"Euphrates corridor governorates: "
    f"{len(euphrates_corridor_admin):,}"
)

print(
    euphrates_corridor_admin[
        [
            "adm1_name",
            "adm1_pcode",
        ]
    ].sort_values(
        "adm1_name"
    )
)

In [ ]:
# 13
# Assign governorate attributes through a spatial join
# 空間結合により報告地点へ県情報を付与する

admin1_join_fields = (
    admin1_web[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "adm1_name": "spatial_adm1_name",
            "adm1_pcode": "spatial_adm1_pcode",
        }
    )
    .copy()
)

joined_flood_locations = gpd.sjoin(
    flood_locations,
    admin1_join_fields,
    how="left",
    predicate="within",
)

print(
    f"Source flood-location features: "
    f"{len(flood_locations):,}"
)

print(
    f"Spatial join result rows: "
    f"{len(joined_flood_locations):,}"
)

print(
    joined_flood_locations[
        [
            "site_id",
            "location",
            "governorate_reported",
            "spatial_adm1_name",
            "spatial_adm1_pcode",
        ]
    ]
)

In [ ]:
# 14
# Validate spatial assignments and reported governorates
# 空間判定結果とCSV記載県の一致を検証する

join_count_by_site = (
    joined_flood_locations
    .groupby(
        "site_id"
    )
    .size()
)

multiply_matched_site_ids = (
    join_count_by_site.loc[
        join_count_by_site > 1
    ]
    .index
    .tolist()
)

if multiply_matched_site_ids:

    raise ValueError(
        "One or more locations were assigned to "
        "multiple governorates: "
        f"{multiply_matched_site_ids}"
    )

unmatched_location_mask = (
    joined_flood_locations[
        "spatial_adm1_name"
    ].isna()
)

if unmatched_location_mask.any():

    unmatched_locations = (
        joined_flood_locations.loc[
            unmatched_location_mask,
            [
                "site_id",
                "location",
                "latitude",
                "longitude",
            ],
        ]
    )

    raise ValueError(
        "One or more flood locations were not assigned "
        "to a governorate:\n"
        f"{unmatched_locations}"
    )

joined_flood_locations[
    "governorate_match"
] = (
    joined_flood_locations[
        "governorate_reported"
    ]
    .eq(
        joined_flood_locations[
            "spatial_adm1_name"
        ]
    )
)

mismatched_governorate_mask = (
    ~joined_flood_locations[
        "governorate_match"
    ]
)

if mismatched_governorate_mask.any():

    mismatched_locations = (
        joined_flood_locations.loc[
            mismatched_governorate_mask,
            [
                "site_id",
                "location",
                "governorate_reported",
                "spatial_adm1_name",
                "spatial_adm1_pcode",
            ],
        ]
    )

    raise ValueError(
        "Reported and spatially assigned governorates "
        "do not match:\n"
        f"{mismatched_locations}"
    )

verified_flood_locations = (
    joined_flood_locations
    .drop(
        columns=[
            "index_right",
        ],
        errors="ignore",
    )
    .copy()
)

print(
    f"Verified flood locations: "
    f"{len(verified_flood_locations):,}"
)

print(
    f"Unmatched locations: "
    f"{int(unmatched_location_mask.sum()):,}"
)

print(
    f"Governorate mismatches: "
    f"{int(mismatched_governorate_mask.sum()):,}"
)

print(
    verified_flood_locations[
        [
            "site_id",
            "location",
            "governorate_reported",
            "spatial_adm1_name",
            "spatial_adm1_pcode",
            "governorate_match",
        ]
    ]
)

In [ ]:
# 15
# Save and verify the validated flood-location dataset
# 検証済みの洪水被害報告地点データを保存し、確認する

validated_output_columns = [
    "site_id",
    "event_id",
    "event_date",
    "governorate_reported",
    "spatial_adm1_name",
    "spatial_adm1_pcode",
    "governorate_match",
    "location",
    "latitude",
    "longitude",
    "source_organisation",
    "source_title",
    "source_url",
    "accessed_date",
    "notes",
    "geometry",
]

validated_flood_locations = (
    verified_flood_locations[
        validated_output_columns
    ]
    .copy()
)

if validated_locations_path.exists():

    validated_locations_path.unlink()

validated_flood_locations.to_file(
    validated_locations_path,
    layer="flood_affected_locations",
    driver="GPKG",
)

saved_flood_locations = gpd.read_file(
    validated_locations_path,
    layer="flood_affected_locations",
)

if len(saved_flood_locations) != len(
    validated_flood_locations
):

    raise ValueError(
        "The saved feature count does not match "
        "the validated feature count."
    )

if saved_flood_locations.crs is None:

    raise ValueError(
        "The saved flood-location dataset "
        "has no defined CRS."
    )

if saved_flood_locations.crs.to_epsg() != 4326:

    raise ValueError(
        "The saved flood-location dataset "
        "does not use EPSG:4326."
    )

if not saved_flood_locations[
    "site_id"
].is_unique:

    raise ValueError(
        "The saved flood-location dataset "
        "contains duplicate site_id values."
    )

print(
    f"Saved validated flood locations: "
    f"{len(saved_flood_locations):,}"
)

print(
    f"Saved CRS: "
    f"{saved_flood_locations.crs}"
)

print(
    f"Saved output: "
    f"{validated_locations_path}"
)

print(
    saved_flood_locations[
        [
            "site_id",
            "location",
            "spatial_adm1_name",
            "spatial_adm1_pcode",
            "source_organisation",
        ]
    ]
)

In [ ]:
# 16
# Prepare the spatial layers for web-map display
# Web地図表示用の空間レイヤーを準備する

# Create geometry-only dissolved boundaries.
# 属性を除外し、ジオメトリのみの結合境界を作成する

country_boundary = (
    admin1_web[
        [
            "geometry",
        ]
    ]
    .dissolve()
    .reset_index(
        drop=True
    )
)

euphrates_corridor_boundary = (
    euphrates_corridor_admin[
        [
            "geometry",
        ]
    ]
    .dissolve()
    .reset_index(
        drop=True
    )
)

euphrates_river_corridor = gpd.clip(
    euphrates_river,
    euphrates_corridor_boundary,
)

water_bodies_corridor = gpd.clip(
    water_bodies_web,
    euphrates_corridor_boundary,
)

if euphrates_river_corridor.empty:

    raise ValueError(
        "No Euphrates River features remain "
        "inside the corridor governorates."
    )

if water_bodies_corridor.empty:

    raise ValueError(
        "No mapped water-body features remain "
        "inside the corridor governorates."
    )

if not euphrates_river_corridor.geometry.is_valid.all():

    raise ValueError(
        "The clipped Euphrates River layer "
        "contains invalid geometries."
    )

if not water_bodies_corridor.geometry.is_valid.all():

    raise ValueError(
        "The clipped water-body layer "
        "contains invalid geometries."
    )

# Preserve the clipped water-body geometries for display.
# クリップ後の水域ジオメトリをそのまま表示用に保持する

water_bodies_display = (
    water_bodies_corridor.copy()
)

corridor_bounds = (
    euphrates_corridor_admin
    .total_bounds
)

map_bounds = [
    [
        corridor_bounds[1],
        corridor_bounds[0],
    ],
    [
        corridor_bounds[3],
        corridor_bounds[2],
    ],
]

print(
    f"Country boundary features: "
    f"{len(country_boundary):,}"
)

print(
    f"Corridor governorates: "
    f"{len(euphrates_corridor_admin):,}"
)

print(
    f"Euphrates River display features: "
    f"{len(euphrates_river_corridor):,}"
)

print(
    f"Water-body display features: "
    f"{len(water_bodies_display):,}"
)

print(
    f"Reported flood locations: "
    f"{len(validated_flood_locations):,}"
)

print(
    f"Map bounds: {map_bounds}"
)

In [ ]:
# 17
# Create the interactive basemap and set the initial extent
# インタラクティブ地図を作成し、初期表示範囲を設定する

map_center_latitude = (
    corridor_bounds[1]
    + corridor_bounds[3]
) / 2

map_center_longitude = (
    corridor_bounds[0]
    + corridor_bounds[2]
) / 2

m = folium.Map(
    location=[
        map_center_latitude,
        map_center_longitude,
    ],
    zoom_start=7,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        "&copy; OpenStreetMap contributors "
        "&copy; CARTO"
    ),
    name="CARTO Light — No Labels",
    overlay=False,
    control=True,
).add_to(m)

m.fit_bounds(
    map_bounds
)

Fullscreen(
    position="topleft",
    title="Open full-screen map",
    title_cancel="Exit full-screen map",
    force_separate_button=True,
).add_to(m)

mini_map_tiles = folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        "&copy; OpenStreetMap contributors "
        "&copy; CARTO"
    ),
)

MiniMap(
    tile_layer=mini_map_tiles,
    position="bottomleft",
    width=150,
    height=100,
    zoom_level_offset=-4,
    toggle_display=True,
).add_to(m)

In [ ]:
# 18
# Add the country and Euphrates corridor boundaries
# 国境およびユーフラテス川流域の県境を追加する

folium.GeoJson(
    country_boundary,
    name="Syria Boundary",
    style_function=lambda feature: {
        "color": "#333333",
        "weight": 2.5,
        "fillColor": "#FFFFFF",
        "fillOpacity": 0.02,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 3,
        "fillOpacity": 0.04,
    },
).add_to(m)

folium.GeoJson(
    euphrates_corridor_admin,
    name="Euphrates Corridor Governorates",
    style_function=lambda feature: {
        "color": "#666666",
        "weight": 1.5,
        "fillColor": "#D9E7E5",
        "fillOpacity": 0.16,
    },
    highlight_function=lambda feature: {
        "color": "#333333",
        "weight": 2,
        "fillOpacity": 0.24,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        localize=True,
        sticky=False,
        labels=True,
    ),
).add_to(m)

In [ ]:
# 19
# Add the mapped water bodies and Euphrates River
# 水域およびユーフラテス川を追加する

folium.GeoJson(
    water_bodies_display,
    name="Mapped Water Bodies",
    style_function=lambda feature: {
        "color": "#5AA9C9",
        "weight": 0.7,
        "fillColor": "#8FD3E8",
        "fillOpacity": 0.45,
    },
    highlight_function=lambda feature: {
        "color": "#287FA3",
        "weight": 1.2,
        "fillOpacity": 0.6,
    },
).add_to(m)

folium.GeoJson(
    euphrates_river_corridor,
    name="Euphrates River",
    style_function=lambda feature: {
        "color": "#0077B6",
        "weight": 3.5,
        "opacity": 0.9,
    },
    highlight_function=lambda feature: {
        "color": "#023E8A",
        "weight": 5,
        "opacity": 1,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "name:en",
            "waterway",
        ],
        aliases=[
            "River:",
            "Feature type:",
        ],
        localize=True,
        sticky=False,
        labels=True,
    ),
).add_to(m)

In [ ]:
# 20
# Add governorate and neighbouring-country labels
# 県名および周辺国ラベルを追加する

governorate_label_group = folium.FeatureGroup(
    name="Governorate Labels",
    show=True,
)

for _, row in euphrates_corridor_admin.iterrows():

    folium.Marker(
        location=[
            row["center_lat"],
            row["center_lon"],
        ],
        icon=folium.DivIcon(
            icon_size=(
                180,
                30,
            ),
            icon_anchor=(
                90,
                15,
            ),
            html=f"""
            <div style="
                width: 180px;
                text-align: center;
                font-size: 15px;
                font-weight: 700;
                color: #303030;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
                pointer-events: none;
            ">
                {row["adm1_name"]}
            </div>
            """,
        ),
    ).add_to(
        governorate_label_group
    )

governorate_label_group.add_to(m)

neighbour_labels = {
    "TÜRKİYE": [
        37.25,
        39.20,
    ],
    "IRAQ": [
        35.45,
        42.55,
    ],
}

neighbour_label_group = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

for country_name, coordinates in neighbour_labels.items():

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            icon_size=(
                160,
                30,
            ),
            icon_anchor=(
                80,
                15,
            ),
            html=f"""
            <div style="
                width: 160px;
                text-align: center;
                font-size: 17px;
                font-weight: 700;
                color: #666666;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
                pointer-events: none;
            ">
                {country_name}
            </div>
            """,
        ),
    ).add_to(
        neighbour_label_group
    )

neighbour_label_group.add_to(m)

In [ ]:
# 21
# Add the reported flood-affected locations
# 洪水被害報告地点を追加する

flood_location_group = folium.FeatureGroup(
    name=(
        "Reported Flood-Affected Locations "
        f"({len(validated_flood_locations):,})"
    ),
    show=True,
)

for _, row in validated_flood_locations.iterrows():

    event_date_text = (
        row["event_date"].strftime(
            "%Y-%m-%d"
        )
        if pd.notna(
            row["event_date"]
        )
        else "Not recorded"
    )

    source_title_text = (
        row["source_title"]
        if pd.notna(
            row["source_title"]
        )
        and str(
            row["source_title"]
        ).strip()
        else "Not recorded"
    )

    source_url_available = (
        pd.notna(
            row["source_url"]
        )
        and str(
            row["source_url"]
        ).strip()
    )

    if source_url_available:

        source_reference_html = f"""
        <a
            href="{row['source_url']}"
            target="_blank"
            rel="noopener noreferrer"
        >
            Open source
        </a>
        """

    else:

        source_reference_html = (
            "Not recorded"
        )

    popup_html = f"""
    <div style="
        width: 280px;
        font-size: 13px;
        line-height: 1.45;
    ">
        <b style="
            font-size: 15px;
            color: #7A271A;
        ">
            {row["location"]}
        </b>

        <hr style="
            margin: 7px 0;
            border: 0;
            border-top: 1px solid #cccccc;
        ">

        <b>Site ID:</b>
        {row["site_id"]}<br>

        <b>Event ID:</b>
        {row["event_id"]}<br>

        <b>Event date:</b>
        {event_date_text}<br>

        <b>Reported governorate:</b>
        {row["governorate_reported"]}<br>

        <b>Spatially verified governorate:</b>
        {row["spatial_adm1_name"]}<br>

        <b>Governorate Pcode:</b>
        {row["spatial_adm1_pcode"]}<br>

        <b>Source organisation:</b>
        {row["source_organisation"]}<br>

        <b>Source title:</b>
        {source_title_text}<br>

        <b>Source reference:</b>
        {source_reference_html}
    </div>
    """

    tooltip_html = f"""
    <div style="
        font-size: 12px;
        line-height: 1.35;
    ">
        <b>{row["location"]}</b><br>
        {row["spatial_adm1_name"]}<br>
        {event_date_text}
    </div>
    """

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x,
        ],
        radius=7,
        color="#7A271A",
        weight=2,
        fill=True,
        fill_color="#D64545",
        fill_opacity=0.9,
        tooltip=folium.Tooltip(
            tooltip_html,
            sticky=False,
        ),
        popup=folium.Popup(
            popup_html,
            max_width=320,
        ),
    ).add_to(
        flood_location_group
    )

flood_location_group.add_to(m)

print(
    f"Flood-location markers added: "
    f"{len(validated_flood_locations):,}"
)

In [ ]:
# 22
# Add the map information and source panel
# 地図の説明、データ範囲および出典情報を追加する

event_date_start = (
    validated_flood_locations[
        "event_date"
    ]
    .min()
    .strftime(
        "%Y-%m-%d"
    )
)

event_date_end = (
    validated_flood_locations[
        "event_date"
    ]
    .max()
    .strftime(
        "%Y-%m-%d"
    )
)

if event_date_start == event_date_end:

    event_date_display = (
        event_date_start
    )

else:

    event_date_display = (
        f"{event_date_start} to "
        f"{event_date_end}"
    )

reported_governorates = ", ".join(
    sorted(
        validated_flood_locations[
            "spatial_adm1_name"
        ]
        .dropna()
        .unique()
    )
)

source_organisations = ", ".join(
    sorted(
        validated_flood_locations[
            "source_organisation"
        ]
        .dropna()
        .unique()
    )
)

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 60px;
    width: 440px;
    min-height: 250px;
    background-color: rgba(255, 255, 255, 0.95);
    color: #222222;
    z-index: 9000;
    font-size: 13px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 13px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="
        font-size: 17px;
    ">
        Syria
    </b>

    <br>

    <span style="
        color: #7A271A;
        font-size: 14px;
        font-weight: 700;
    ">
        Reported Flood-Affected Locations
        Along the Euphrates
    </span>

    <div style="
        margin-top: 8px;
        line-height: 1.4;
        color: #333333;
    ">
        This map displays the flood-affected locations
        currently recorded in the project CSV dataset.
        The records represent reported point locations
        and do not describe the complete spatial extent
        of flooding.
    </div>

    <div style="
        margin-top: 10px;
        padding-top: 8px;
        line-height: 1.4;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        <b>Event date:</b>
        {event_date_display}<br>

        <b>Recorded events:</b>
        {event_count:,}<br>

        <b>Reported locations:</b>
        {len(validated_flood_locations):,}<br>

        <b>Spatially verified locations:</b>
        {int(validated_flood_locations["governorate_match"].sum()):,}<br>

        <b>Reported governorates:</b>
        {reported_governorates}<br>

        <b>Source organisations recorded:</b>
        {source_organisations}
    </div>

    <div style="
        margin-top: 9px;
        padding-top: 8px;
        font-size: 11px;
        line-height: 1.4;
        color: #666666;
        border-top: 1px solid #cccccc;
    ">
        Location data:
        user-maintained project CSV<br>

        Hydrography:
        project river and water-body datasets<br>

        Method:
        CSV validation / Point creation /
        Spatial join / Attribute verification<br>

        Ar-Raqqa is displayed as geographical context;
        the current CSV contains no reported location
        in that governorate.<br>

        Detailed source titles, URLs and access dates
        remain to be added to the CSV.
    </div>
</div>
"""

m.get_root().html.add_child(
    Element(
        information_panel_html
    )
)

In [ ]:
# 23
# Add the map legend and layer control
# 地図の凡例およびレイヤーコントロールを追加する

legend_html = f"""
<div style="
    position: fixed;
    right: 30px;
    bottom: 30px;
    width: 300px;
    background-color: rgba(255, 255, 255, 0.95);
    color: #222222;
    z-index: 9000;
    font-size: 12px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.22);
">
    <b style="
        font-size: 14px;
    ">
        Map Legend
    </b>

    <div style="
        margin-top: 10px;
        line-height: 1.7;
    ">
        <span style="
            display: inline-block;
            width: 28px;
            height: 14px;
            margin-right: 9px;
            vertical-align: middle;
            background-color: rgba(217, 231, 229, 0.55);
            border: 2px solid #666666;
        "></span>
        Euphrates corridor governorate
        <br>

        <span style="
            display: inline-block;
            width: 28px;
            height: 14px;
            margin-right: 9px;
            vertical-align: middle;
            background-color: rgba(143, 211, 232, 0.6);
            border: 1px solid #5AA9C9;
        "></span>
        Mapped water body
        <br>

        <span style="
            display: inline-block;
            width: 30px;
            height: 0;
            margin-right: 9px;
            vertical-align: middle;
            border-top: 4px solid #0077B6;
        "></span>
        Euphrates River
        <br>

        <span style="
            display: inline-block;
            width: 12px;
            height: 12px;
            margin-left: 8px;
            margin-right: 17px;
            vertical-align: middle;
            background-color: #D64545;
            border: 2px solid #7A271A;
            border-radius: 50%;
        "></span>
        Reported flood-affected location
    </div>

    <div style="
        margin-top: 10px;
        padding-top: 8px;
        line-height: 1.4;
        color: #666666;
        border-top: 1px solid #aaaaaa;
    ">
        Reported locations:
        <b>{len(validated_flood_locations):,}</b><br>

        Recorded events:
        <b>{event_count:,}</b><br>

        Map CRS:
        <b>EPSG:4326</b><br>

        Point symbols indicate reported locations,
        not flood extent.
    </div>
</div>
"""

m.get_root().html.add_child(
    Element(
        legend_html
    )
)

folium.LayerControl(
    position="topright",
    collapsed=False,
).add_to(m)

In [ ]:
# 24
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m